# Notebook 5: revision experiments

Answers the remaining reviewer concerns (1, 3, 5, 6, and 7). Run it **after Notebooks 1, 3, and 4 have finished**, in the same `Python (iomt)` kernel, from the folder that holds the notebooks, `cleaned_data/`, `Train/`, `results_revised/checkpoint.pkl`, and `results_supplementary/supp_checkpoint.pkl`.

| Part | Concern | Needs | Approximate time |
|---|---|---|---|
| A. Size-controlled paired tests | 3, 6 | saved Notebook 4 results only | seconds |
| B. IAT offset removal and file-disjoint test | 1, 4 | raw `Train/` SYN part files | 5–15 min |
| C. Timing jitter on IAT; per-class recall after threshold | 1, 5 | cleaned data | 10–20 min |
| F. Constraint checks; correlated, constraint-preserving noise | 2nd-round C | cleaned data | 20–40 min |
| D. Search budget (20, 40, 60 generations, runs 1–3), run last | 7 | cleaned data | 1–2 h |

Results are written to `results_revision/revision_results.xlsx` and `results_revision/revision_summary.txt`. Archives from Part D are saved as `.npz` files.

In [1]:
# Load and run every code cell of Notebook 4 in this notebook, so its functions and data are available here.
# Its main loop skips runs already saved, so this only reloads data and rebuilds the perturbed test copies.
# (This replaces the Jupyter run command, which needs the nbformat package.)
import json as _json
_nb4 = _json.load(open("4_Supplementary_Analyses.ipynb", encoding="utf-8"))
for _cell in _nb4["cells"]:
    if _cell["cell_type"] == "code":
        _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
        exec(compile(_src, "4_Supplementary_Analyses.ipynb", "exec"), globals())
print("Notebook 4 loaded.")

REV_OUT = "results_revision"; os.makedirs(REV_OUT, exist_ok=True)
P = "Proposed: NSGA-II (3-obj) + consensus"
assert len(state["runs"]) == N_RUNS, "Notebook 4 must finish all runs first"

def paired(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); n = len(d)
    h = stats.t.ppf(0.975, n - 1) * d.std(ddof=1) / np.sqrt(n)
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return {"mean diff": d.mean(), "95% CI low": d.mean() - h, "95% CI high": d.mean() + h,
            "runs favoring first": int((d > 0).sum()), "Wilcoxon p": p}

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); top = 0.0
    for r, i in enumerate(o):
        top = max(top, (m - r) * p[i]); adj[i] = min(1.0, top)
    return adj
print("Ready:", len(state["runs"]), "runs loaded")

Using checkpoint: results_revised\checkpoint.pkl
Windows-10-10.0.26300-SP0 | CPU: Intel64 Family 6 Model 142 Stepping 10, GenuineIntel | cores: 8 | Python 3.11.17 | scikit-learn 1.9.1 | pymoo 0.6.2 | numpy 2.4.6 | pandas 3.0.6
Runs in checkpoint: 10
Range-coded perturbed features: 20 | meaning-coded perturbed features: 19
Perturbed under range coding but not meaning coding: ['Protocol Type', 'Duration']
Perturbed under meaning coding but not range coding: ['Variance']
Perturbation sets: ['gauss_5', 'gauss_10', 'gauss_20', 'semantic_gauss_10', 'uniform', 'laplace', 'missing_10', 'missing_20', 'quantise']
Functions ready.
Saved progress found for runs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'max_abs_diff_clean_F1': np.float64(0.0), 'max_abs_diff_F1_noise10': np.float64(0.0), 'proposed_subset_reproduced_runs': 10, 'runs': 10}
{'mean difference': np.float64(-0.00495226344328078), '90% CI low': np.float64(-0.008142615247464569), '90% CI high': np.float64(-0.0017619116390969916), 'margin': 0.01, 'W

## A. Size-controlled consensus versus single solutions (Table 6)

In [2]:
sens = pd.DataFrame([r for R in state["runs"].values() for r in R["sens"]])
rows_s = pd.DataFrame([r for R in state["runs"].values() for r in R["rows"]])
def per_run(method, col):
    return rows_s[rows_s.method == method].set_index("run")[col].sort_index()

out = []
for (tau, theta), g in sens.groupby(["tau", "theta"]):
    g = g.set_index("run").sort_index()
    for comp in ["NSGA-II (3-obj)", "NSGA-II (2-obj)", "RFE (top-k)", "MI (top-k)", "GA single-objective", "All features"]:
        for col in ["F1_gauss_10", "F1_semantic_gauss_10", "macro_F1"]:
            r = paired(g[col].values, per_run(comp, col).loc[g.index].values)
            out.append({"tau": tau, "theta": theta, "mean size": g["n_features"].mean(), "vs": comp, "metric": col, **r})
A = pd.DataFrame(out)
A["Holm p (9 settings)"] = A.groupby(["vs", "metric"])["Wilcoxon p"].transform(holm)
print(A[(A.theta == 0.7) & (A.vs == "NSGA-II (3-obj)")].round(4).to_string(index=False))

  tau  theta  mean size              vs               metric  mean diff  95% CI low  95% CI high  runs favoring first  Wilcoxon p  Holm p (9 settings)
0.005    0.7       15.9 NSGA-II (3-obj)          F1_gauss_10     0.0204     -0.0019       0.0428                    7      0.1055               0.3164
0.005    0.7       15.9 NSGA-II (3-obj) F1_semantic_gauss_10     0.0144     -0.0097       0.0384                    8      0.1055               0.3164
0.005    0.7       15.9 NSGA-II (3-obj)             macro_F1    -0.0003     -0.0091       0.0084                    5      0.9219               1.0000
0.010    0.7       15.5 NSGA-II (3-obj)          F1_gauss_10     0.0221     -0.0056       0.0498                    5      0.2324               0.4648
0.010    0.7       15.5 NSGA-II (3-obj) F1_semantic_gauss_10     0.0179     -0.0089       0.0446                    6      0.2324               0.4648
0.010    0.7       15.5 NSGA-II (3-obj)             macro_F1    -0.0029     -0.0097       0.00

## B. IAT and recording sessions (raw SYN part files, as in test G1)

In [3]:
parts = []
for f in sorted(glob.glob(os.path.join(RAW_TRAIN, "TCP_IP-D*oS-SYN[1-4]_train.pcap.csv"))):
    m_ = re.search(r"(TCP_IP-D?DoS-SYN)(\d)_train", os.path.basename(f))
    df = pd.read_csv(f).sample(20000, random_state=0); df["cls"] = m_.group(1); df["part"] = int(m_.group(2)); parts.append(df)
P4 = pd.concat(parts, ignore_index=True)
P4["IAT_centered"] = P4["IAT"] - P4.groupby(["cls", "part"])["IAT"].transform("median")

# B1. Does IAT still identify the part file once the per-file offset is removed? Chance = 0.25.
cv5 = StratifiedKFold(5, shuffle=True, random_state=0); b1 = []
for cls, g in P4.groupby("cls"):
    for name in ["IAT", "IAT_centered"]:
        pred = cross_val_predict(RandomForestClassifier(100, random_state=0, n_jobs=-1), g[[name]].values,
                                 g["part"].values, cv=cv5)
        b1.append({"class": cls, "predictor": name, "accuracy predicting part file": float(np.mean(pred == g["part"].values))})
B1 = pd.DataFrame(b1); print(B1.round(3).to_string(index=False))

# B2. DDoS-SYN vs DoS-SYN: random split versus leaving out one part number of both classes at a time.
others = [c for c in FEATURES_ALL if c != "IAT"]
y_b = (P4["cls"] == "TCP_IP-DDoS-SYN").astype(int).values; b2 = []
for name, cols in [("without IAT", others), ("with IAT", others + ["IAT"]), ("IAT only", ["IAT"]),
                   ("centered IAT only", ["IAT_centered"])]:
    X_b = P4[cols].values
    rnd = cross_val_predict(RandomForestClassifier(100, random_state=0, n_jobs=-1), X_b, y_b,
                            cv=StratifiedKFold(4, shuffle=True, random_state=0))
    grp = np.empty_like(y_b)
    for k in range(1, 5):
        te = P4["part"].values == k
        grp[te] = RandomForestClassifier(100, random_state=0, n_jobs=-1).fit(X_b[~te], y_b[~te]).predict(X_b[te])
    b2.append({"features": name, "random-split macro-F1": f1_score(y_b, rnd, average="macro"),
               "file-disjoint macro-F1": f1_score(y_b, grp, average="macro")})
B2 = pd.DataFrame(b2); B2["gap"] = B2["random-split macro-F1"] - B2["file-disjoint macro-F1"]
print(B2.round(3).to_string(index=False))

          class    predictor  accuracy predicting part file
TCP_IP-DDoS-SYN          IAT                          0.972
TCP_IP-DDoS-SYN IAT_centered                          0.434
 TCP_IP-DoS-SYN          IAT                          0.970
 TCP_IP-DoS-SYN IAT_centered                          0.506
         features  random-split macro-F1  file-disjoint macro-F1   gap
      without IAT                  0.929                   0.776 0.153
         with IAT                  1.000                   1.000 0.000
         IAT only                  1.000                   1.000 0.000
centered IAT only                  0.603                   0.467 0.136


## C. Timing-consistent jitter on IAT, and per-class recall after the benign threshold

In [4]:
# C1. Additive jitter in IAT's own units, scaled to the median within-class IQR of the training data.
q = train_all.groupby("Label")["IAT"].quantile; s0 = float((q(0.75) - q(0.25)).median())
X_tr, y_tr = sample_train(0); ia = COL["IAT"]; all_cols = list(range(len(FEATURES_ALL)))
mA = make_forest(FINAL_TREES, 0).fit(X_tr[:, all_cols], y_tr)
mI = make_forest(FINAL_TREES, 0).fit(X_tr[:, [ia]], y_tr)
rng = np.random.default_rng(42); c1 = []
for scale in [0, 0.1, 0.5, 1, 2, 5]:
    Xj = X_test.copy(); Xj[:, ia] = np.maximum(0, Xj[:, ia] + rng.normal(0, scale * s0, len(Xj)))
    c1.append({"jitter SD (x median within-class IQR)": scale, "jitter SD (IAT units)": scale * s0,
               "All + IAT macro-F1": fast_macro_f1(yt_i, to_idx(mA.predict(Xj))),
               "IAT-only macro-F1": fast_macro_f1(yt_i, to_idx(mI.predict(Xj[:, [ia]])))})
C1 = pd.DataFrame(c1); print("median within-class IQR of IAT:", round(s0, 1)); print(C1.round(3).to_string(index=False))
del mA, mI; gc.collect()

# C2. Per-class recall of the proposed subsets before and after the out-of-bag benign threshold (all runs).
c2 = []
for run in range(N_RUNS):
    X_tr, y_tr = sample_train(run)
    cols = [COL[f] for f in state["runs"][run]["subsets"][P]]
    m = make_forest(FINAL_TREES, run, oob=True).fit(X_tr[:, cols], y_tr)
    bi = list(m.classes_).index("Benign")
    t = float(np.nanquantile(m.oob_decision_function_[np.asarray(y_tr) == "Benign", bi], 0.05))
    prob = m.predict_proba(X_test[:, cols]); att = prob.copy(); att[:, bi] = -1
    p0 = to_idx(m.classes_[prob.argmax(1)]); pt = to_idx(m.classes_[np.where(prob[:, bi] >= t, bi, att.argmax(1))])
    for c, ci in CIDX.items():
        s_ = yt_i == ci
        c2.append({"run": run, "class": c, "recall default": float(np.mean(p0[s_] == ci)),
                   "recall threshold": float(np.mean(pt[s_] == ci))})
    del m; gc.collect()
C2 = pd.DataFrame(c2).groupby("class")[["recall default", "recall threshold"]].agg(["mean", "std"])
print(C2.round(3))

median within-class IQR of IAT: 169362896.0
 jitter SD (x median within-class IQR)  jitter SD (IAT units)  All + IAT macro-F1  IAT-only macro-F1
                                   0.0           0.000000e+00               0.885              0.753
                                   0.1           1.693629e+07               0.600              0.205
                                   0.5           8.468145e+07               0.600              0.134
                                   1.0           1.693629e+08               0.598              0.109
                                   2.0           3.387258e+08               0.595              0.095
                                   5.0           8.468145e+08               0.592              0.082
                        recall default        recall threshold       
                                  mean    std             mean    std
class                                                                
ARP_Spoofing                     0.845 

## F. Physical constraints and a correlated, constraint-preserving perturbation (second-round concern C)

In [5]:
# F1. Do any features take negative values, and how often does the independent noise break Min <= AVG <= Max?
mins = train_all[FEATURES_ALL].min(); neg = mins[mins < 0]
print("features with negative training values:", dict(neg) if len(neg) else "none")
iMin, iAVG, iMax = COL["Min"], COL["AVG"], COL["Max"]
def order_broken(X): return (X[:, iMin] > X[:, iAVG] + 1e-9) | (X[:, iAVG] > X[:, iMax] + 1e-9)
f1 = [{"test copy": "clean", "Min<=AVG<=Max broken (share of rows)": float(order_broken(X_test).mean()),
       "rows with a value clipped to 0 (share)": 0.0}]
for key in ["gauss_5", "gauss_10", "gauss_20", "semantic_gauss_10", "uniform", "laplace"]:
    Xn = PERT[key][0]
    f1.append({"test copy": key, "Min<=AVG<=Max broken (share of rows)": float(order_broken(Xn).mean()),
               "rows with a value clipped to 0 (share)": float(((Xn == 0) & (X_test != 0)).any(axis=1).mean())})
F1 = pd.DataFrame(f1); print(F1.round(4).to_string(index=False))

# F2. Correlated noise: one shared factor per record for each physical group, so relations between features hold.
#   lengths (packet-length statistics and header length) x (1+e_len); Covariance x (1+e_len)^2; Variance (a ratio) unchanged
#   counts x (1+e_cnt); Weight (a product of counts) x (1+e_cnt)^2; Rate x (1+e_rate)
#   indicators, Protocol Type, Duration (TTL), and IAT stay fixed
def grp(names): return [COL[c] for c in names if c in COL]
G_LEN = grp(["Header_Length", "Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "Magnitue", "Magnitude", "Radius"])
G_LEN2 = grp(["Covariance"]); G_CNT = grp(["ack_count", "syn_count", "fin_count", "rst_count", "Number"])
G_CNT2 = grp(["Weight"]); G_RATE = grp(["Rate"])
def correlated(X, rng, s=0.10):
    Xn = X.astype("float64").copy(); n = len(X)
    e_len, e_cnt, e_rate = (np.clip(1 + rng.normal(0, s, n), 0, None) for _ in range(3))
    Xn[:, G_LEN] *= e_len[:, None]; Xn[:, G_LEN2] *= (e_len ** 2)[:, None]
    Xn[:, G_CNT] *= e_cnt[:, None]; Xn[:, G_CNT2] *= (e_cnt ** 2)[:, None]; Xn[:, G_RATE] *= e_rate[:, None]
    return Xn.astype("float32")
PERT_C = [correlated(X_test, np.random.default_rng(11000 + d)) for d in range(NOISE_DRAWS)]
print("correlated copy keeps Min<=AVG<=Max:", not order_broken(PERT_C[0]).any())

methods_F = ORIG + ["Random (k)", "MI bootstrap consensus (k)", "GA single-objective"]
f2 = []
for run in range(N_RUNS):
    X_tr, y_tr = sample_train(run)
    for mth in methods_F:
        cols = [COL[f] for f in state["runs"][run]["subsets"][mth]]
        m = make_forest(FINAL_TREES, run).fit(X_tr[:, cols], y_tr)
        f2.append({"run": run, "method": mth, "n_features": len(cols),
                   "macro_F1": fast_macro_f1(yt_i, to_idx(m.predict(X_test[:, cols]))),
                   "F1_correlated_10": float(np.mean([fast_macro_f1(yt_i, to_idx(m.predict(Xc[:, cols]))) for Xc in PERT_C]))})
        del m
    gc.collect(); print(f"run {run + 1}/{N_RUNS} done", flush=True)
F2runs = pd.DataFrame(f2)
F2 = F2runs.groupby("method")[["n_features", "macro_F1", "F1_correlated_10"]].mean().sort_values("F1_correlated_10", ascending=False)
w = F2runs.pivot(index="run", columns="method", values="F1_correlated_10")
F2p = pd.DataFrame([{"vs": c, **paired(w[P], w[c])} for c in methods_F if c != P])
F2p["Holm p"] = holm(F2p["Wilcoxon p"].values)
print(F2.round(4)); print(F2p.round(4).to_string(index=False))

features with negative training values: none
        test copy  Min<=AVG<=Max broken (share of rows)  rows with a value clipped to 0 (share)
            clean                                0.0000                                     0.0
          gauss_5                                0.4958                                     0.0
         gauss_10                                0.5813                                     0.0
         gauss_20                                0.6599                                     0.0
semantic_gauss_10                                0.5792                                     0.0
          uniform                                0.5858                                     0.0
          laplace                                0.5657                                     0.0
correlated copy keeps Min<=AVG<=Max: True
run 1/10 done
run 2/10 done
run 3/10 done
run 4/10 done
run 5/10 done
run 6/10 done
run 7/10 done
run 8/10 done
run 9/10 done
run 10/10 done
    

## D. Search budget: 20, 40, and 60 generations on runs 1–3

In [6]:
D = []; GA_GEN_DEFAULT = GA_GEN
for run in [0, 1, 2]:
    X_tr, y_tr = sample_train(run); folds = make_folds(X_tr, y_tr, run); ref = None
    saved = set(state["runs"][run]["subsets"][P])
    for G in [20, 40, 60]:
        GA_GEN = G
        t1 = time.perf_counter(); res, aX, aF, hv = run_search(folds, run, 3); secs = time.perf_counter() - t1
        np.savez_compressed(os.path.join(REV_OUT, f"archive_run{run + 1}_G{G}.npz"), X=aX, F=aF, hv=np.array(hv))
        cols, nn = pick_consensus(aF, aX, F1_TOLERANCE, CONSENSUS)
        r, _ = evaluate_full(cols, X_tr, y_tr, run, extended=False)
        s = set(FEATURES_ALL[i] for i in cols); ref = s if G == 20 else ref
        D.append({"run": run + 1, "generations": G, "search s": secs, "archive size": len(aX),
                  "final population hypervolume": hv[-1], "near-optimal subsets": nn, "consensus size": len(cols),
                  "same as saved subset": s == saved, "Jaccard with G=20 subset": len(s & ref) / len(s | ref),
                  "macro_F1": r["macro_F1"], "F1_gauss_10": r["F1_gauss_10"],
                  "F1_semantic_gauss_10": r["F1_semantic_gauss_10"]})
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in D[-1].items()}, flush=True)
GA_GEN = GA_GEN_DEFAULT
D = pd.DataFrame(D); print(D.round(4).to_string(index=False))

{'run': 1, 'generations': 20, 'search s': 127.5677, 'archive size': 480, 'final population hypervolume': 0.7138, 'near-optimal subsets': 7, 'consensus size': 15, 'same as saved subset': True, 'Jaccard with G=20 subset': 1.0, 'macro_F1': 0.786, 'F1_gauss_10': 0.7002, 'F1_semantic_gauss_10': 0.7131}
{'run': 1, 'generations': 40, 'search s': 249.4118, 'archive size': 960, 'final population hypervolume': 0.803, 'near-optimal subsets': 5, 'consensus size': 21, 'same as saved subset': False, 'Jaccard with G=20 subset': 0.5652, 'macro_F1': 0.7889, 'F1_gauss_10': 0.7202, 'F1_semantic_gauss_10': 0.7226}
{'run': 1, 'generations': 60, 'search s': 377.8436, 'archive size': 1439, 'final population hypervolume': 0.7994, 'near-optimal subsets': 1, 'consensus size': 17, 'same as saved subset': False, 'Jaccard with G=20 subset': 0.6842, 'macro_F1': 0.7897, 'F1_gauss_10': 0.7109, 'F1_semantic_gauss_10': 0.7211}
{'run': 2, 'generations': 20, 'search s': 225.178, 'archive size': 480, 'final population hyp

## E. Save results

In [ ]:
with pd.ExcelWriter(os.path.join(REV_OUT, "revision_results.xlsx")) as xw:
    A.to_excel(xw, sheet_name="A_size_controlled", index=False)
    B1.to_excel(xw, sheet_name="B1_IAT_part_file", index=False)
    B2.to_excel(xw, sheet_name="B2_file_disjoint", index=False)
    C1.to_excel(xw, sheet_name="C1_IAT_jitter", index=False)
    C2.to_excel(xw, sheet_name="C2_recall_threshold")
    D.to_excel(xw, sheet_name="D_budget", index=False)
    F1.to_excel(xw, sheet_name="F1_constraints", index=False)
    F2.to_excel(xw, sheet_name="F2_correlated_means")
    F2p.to_excel(xw, sheet_name="F2_correlated_paired", index=False)
    F2runs.to_excel(xw, sheet_name="F2_correlated_runs", index=False)
with open(os.path.join(REV_OUT, "revision_summary.txt"), "w") as fh:
    fh.write(f"Environment: Python {platform.python_version()} | scikit-learn {sklearn.__version__} | pymoo {pymoo.__version__}\n\n")
    for name, df in [("A (theta = 0.7 vs NSGA-II 3-obj)", A[(A.theta == 0.7) & (A.vs == "NSGA-II (3-obj)")]),
                     ("B1", B1), ("B2", B2), ("C1", C1), ("C2", C2), ("D", D), ("F1", F1), ("F2", F2), ("F2 paired", F2p)]:
        fh.write(name + "\n" + df.round(4).to_string() + "\n\n")
print("Saved to", REV_OUT)